# Data Preprocessing
This notebook begins by constructing the data tensor used throughout all downstream modeling experiments. 

The raw inputs consist of two AnnData objects:
- Gene-level expression matrix (`bulk_processed_genes.h5ad`)
- Transcript-level (isoform) expression matrix (`bulk_processed_transcripts.h5ad`)
These files must be placed in: `$BLACKHOLE/<username>/`

Because these matrices are stored in sparse or backed-on-disk formats, we convert them into dense float32 NumPy arrays, followed by PyTorch tensors. Gene expression is transformed using log1p, while isoform expression is kept in absolute scale (the prediction target).

In addition to expression matrices, we retain essential annotation metadata, including:
- gene and transcript identifiers
- gene→transcript mappings
- number of isoforms per gene
- indices linking transcripts and genes
All components are finally saved into a single `data.pt` file, which ensures reproducible loading across all model scripts.


In [ ]:
import anndata as ad
import numpy as np
import torch
from scipy import sparse
import os
import argparse


RUNDIR = os.path.join(os.environ["BLACKHOLE"], os.environ["USER"])
GENE_H5AD = f"{RUNDIR}/bulk_processed_genes.h5ad"
TX_H5AD   = f"{RUNDIR}/bulk_processed_transcripts.h5ad"
OUT_PT    = f"{RUNDIR}/data.pt"

print("Starting data preprocessing...")
def to_dense_f32(x):
    if hasattr(x, "to_memory"):     
        x = x.to_memory()        
    if sparse.issparse(x):
        return x.astype(np.float32).toarray()
    return np.asarray(x, dtype=np.float32)

# Load and read data (takes around 8 min)
gene_ad = ad.read_h5ad(GENE_H5AD, backed = 'r')

print(gene_ad)

tx_ad   = ad.read_h5ad(TX_H5AD, backed ='r')


print(tx_ad)

# Transform data to tensors
X_gene = torch.tensor(to_dense_f32(gene_ad.X), dtype=torch.float32)  # (N, G)
X_tx   = torch.tensor(to_dense_f32(tx_ad.X),   dtype=torch.float32)  # (N, I)

# Get gene and transcripts IDs
gene_ids = (gene_ad.var["gene_id"].astype(str).tolist()
            if "gene_id" in gene_ad.var.columns else gene_ad.var_names.astype(str).tolist())
tx_ids   = tx_ad.var_names.astype(str).tolist()

# log1p genes
Xg_log1p = torch.log1p(X_gene)
Y_tx     = X_tx  # targets = absolute isoform expression, all transcripts kept

print(f"Samples: {Xg_log1p.shape[0]}")
print(f"Genes:   {Xg_log1p.shape[1]}")
print(f"Isoforms:{Y_tx.shape[1]}")

# After loading everything, we also include some more stuff from gene_ad.uns
gene_to_transcripts   = gene_ad.uns["gene_to_transcripts"]
gene_n_transcripts    = gene_ad.uns["gene_n_transcripts"]
multi_isoform_genes   = gene_ad.uns["multi_isoform_genes"]
single_isoform_genes  = gene_ad.uns["single_isoform_genes"]
transcript_ids        = gene_ad.uns["transcript_ids"]
transcript_id_to_index = gene_ad.uns["transcript_id_to_index"]
transcript_mapping    = gene_ad.uns["transcript_mapping"]

torch.save({
    "X_gene": X_gene,
    "Xg_log1p": Xg_log1p,
    "Y_tx": Y_tx,
    "gene_ids": gene_ids,
    "tx_ids": tx_ids,

    "gene_to_transcripts":   gene_to_transcripts,
    "gene_n_transcripts":    gene_n_transcripts,
    "multi_isoform_genes":   multi_isoform_genes,
    "single_isoform_genes":  single_isoform_genes,
    "transcript_ids":        transcript_ids,
    "transcript_id_to_index": transcript_id_to_index,
    "transcript_mapping":    transcript_mapping,
}, OUT_PT)





# Inspecting the Processed Dataset (`data.pt`)
The following cell loads this file and prints basic information about its contents, including tensor shapes, ID consistency, sparsity statistics, and checks for NaN/Inf values.
For reproducibility, the notebook assumes the file is located in `$BLACKHOLE/<username>/`.


In [ ]:

def parse_args():
    parser = argparse.ArgumentParser(
        description="Inspect contents of a data.pt file (tensors, shapes, IDs, mappings, QC checks)."
    )
    parser.add_argument(
        "--path",
        type=str,
        default=None,
        help="Full path to data.pt. If not given, will use $BLACKHOLE/$USER/data.pt.",
    )
    return parser.parse_args()


def resolve_default_path():
    blackhole = os.environ.get("BLACKHOLE")
    user = os.environ.get("USER")

    if blackhole is None or user is None:
        raise RuntimeError(
            "BLACKHOLE and/or USER environment variables are not set. "
            "Set them or use --path /full/path/to/data.pt"
        )

    rundir = os.path.join(blackhole, user)
    return os.path.join(rundir, "data.pt")


def main():
    args = parse_args()

    if args.path is not None:
        pt_path = args.path
    else:
        pt_path = resolve_default_path()

    print(f"\nLoading: {pt_path}")
    data = torch.load(pt_path, weights_only=False)

    # -------------------------------------------------------
    # Basic structure: keys
    # -------------------------------------------------------
    print("\n=== Keys in data.pt ===")
    for k in data.keys():
        print(f" - {k}")

    # Extract tensors and metadata
    Xg = data["Xg_log1p"]
    Y = data["Y_tx"]
    gene_ids = data["gene_ids"]
    tx_ids = data["tx_ids"]
    g2t = data["gene_to_transcripts"]
    t2i = data["transcript_id_to_index"]

    # -------------------------------------------------------
    # Shapes
    # -------------------------------------------------------
    print("\n=== Tensor shapes ===")
    print("Xg_log1p (genes)      :", tuple(Xg.shape))
    print("Y_tx (isoforms)       :", tuple(Y.shape))

    # -------------------------------------------------------
    # (9) Consistency checks
    # -------------------------------------------------------
    print("\n=== Consistency checks ===")
    print("Gene count matches IDs:",
          Xg.shape[1] == len(gene_ids))
    print("Isoform count matches IDs:",
          Y.shape[1] == len(tx_ids))

    # transcript index consistency
    t2i_ok = all(t2i[tx_ids[i]] == i for i in range(len(tx_ids)))
    print("Transcript index consistency:", t2i_ok)

    # -------------------------------------------------------
    # (1) Distribution of transcripts per gene
    # -------------------------------------------------------
    print("\n=== Transcript count per gene ===")
    isoform_counts = [len(g2t[g]) for g in g2t]
    isoform_counts = np.array(isoform_counts)

    print("Genes:", len(isoform_counts))
    print("Min isoforms per gene:", isoform_counts.min())
    print("Max isoforms per gene:", isoform_counts.max())
    print("Mean isoforms per gene:", isoform_counts.mean())
    print("Genes with 1 isoform:", np.sum(isoform_counts == 1))
    print("Genes with 2 isoforms:", np.sum(isoform_counts == 2))
    print("Genes with ≥3 isoforms:", np.sum(isoform_counts >= 3))

    # -------------------------------------------------------
    # (2) NaN / Inf checks
    # -------------------------------------------------------
    print("\n=== NaN / Inf checks ===")
    print("Xg_log1p NaN:", torch.isnan(Xg).any().item())
    print("Xg_log1p Inf:", torch.isinf(Xg).any().item())
    print("Y_tx NaN:", torch.isnan(Y).any().item())
    print("Y_tx Inf:", torch.isinf(Y).any().item())

    # -------------------------------------------------------
    # (3) Sparsity statistics
    # -------------------------------------------------------
    print("\n=== Sparsity statistics ===")
    Xg_zero = (Xg == 0).sum().item()
    Y_zero = (Y == 0).sum().item()

    Xg_total = Xg.numel()
    Y_total = Y.numel()

    print(f"Xg_log1p sparsity: {Xg_zero / Xg_total:.4f} ({Xg_zero}/{Xg_total})")
    print(f"Y_tx sparsity:     {Y_zero / Y_total:.4f} ({Y_zero}/{Y_total})")

    # -------------------------------------------------------
    # (8) Genes/transcripts with zero total expression
    # -------------------------------------------------------
    print("\n=== Zero-expression genes & isoforms ===")
    zero_gene = (Xg.sum(dim=0) == 0).sum().item()
    zero_tx = (Y.sum(dim=0) == 0).sum().item()

    print("Zero-expression genes:   ", zero_gene)
    print("Zero-expression isoforms:", zero_tx)

    # -------------------------------------------------------
    # (6) Detailed transcript index consistency example
    # -------------------------------------------------------
    example_tx = tx_ids[0]
    example_index = t2i[example_tx]

    print("\n=== Transcript ID → index example ===")
    print("Transcript ID:", example_tx)
    print("Mapped index :", example_index)
    print("Vector length consistency:",
          example_index < Y.shape[1])

    print("\n=== DONE inspecting data.pt ===\n")


if __name__ == "__main__":
    main()
